<a href="https://colab.research.google.com/github/zkoymen/yolo26-cnn-architecture-lab/blob/main/experiments/02_c3k2_depth_ablation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q -U ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 10.9 MB/s eta 0:00:00


In [3]:
import copy
import yaml
import torch

from ultralytics import YOLO

print("GPU:", torch.cuda.get_device_name(0))

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
GPU: NVIDIA L4


In [4]:
baseline = YOLO("yolo26s.pt")

baseline_params = sum(
    p.numel() for p in baseline.model.parameters()
)

print(f"Baseline parameters: {baseline_params:,}")

for i, layer in enumerate(baseline.model.model):
    print(
        f"{i:02d} | "
        f"{layer.__class__.__name__:10s} | "
        f"params={sum(p.numel() for p in layer.parameters()):,}"
    )

Baseline parameters: 10,009,784
00 | Conv       | params=928
01 | Conv       | params=18,560
02 | C3k2       | params=26,080
03 | Conv       | params=147,712
04 | C3k2       | params=103,360
05 | Conv       | params=590,336
06 | C3k2       | params=346,112
07 | Conv       | params=1,180,672
08 | C3k2       | params=1,380,352
09 | SPPF       | params=656,896
10 | C2PSA      | params=990,976
11 | Upsample   | params=0
12 | Concat     | params=0
13 | C3k2       | params=477,184
14 | Upsample   | params=0
15 | Concat     | params=0
16 | C3k2       | params=136,192
17 | Conv       | params=147,712
18 | Concat     | params=0
19 | C3k2       | params=378,880
20 | Conv       | params=590,336
21 | Concat     | params=0
22 | C3k2       | params=1,843,712
23 | Detect     | params=993,784


In [5]:
cfg = copy.deepcopy(baseline.model.yaml)

for i, layer in enumerate(cfg["backbone"]):
    print(i, layer)


# it says 512 channels but in reality due to scaling in yolov26-small,
# it outputs 256 channel in layer 4

0 [-1, 1, 'Conv', [64, 3, 2]]
1 [-1, 1, 'Conv', [128, 3, 2]]
2 [-1, 2, 'C3k2', [256, False, 0.25]]
3 [-1, 1, 'Conv', [256, 3, 2]]
4 [-1, 2, 'C3k2', [512, False, 0.25]]
5 [-1, 1, 'Conv', [512, 3, 2]]
6 [-1, 2, 'C3k2', [512, True]]
7 [-1, 1, 'Conv', [1024, 3, 2]]
8 [-1, 2, 'C3k2', [1024, True]]
9 [-1, 1, 'SPPF', [1024, 5, 3, True]]
10 [-1, 2, 'C2PSA', [1024]]


In [6]:
import copy
import yaml
from ultralytics import YOLO

cfg = copy.deepcopy(baseline.model.yaml)

print("Before:", cfg["backbone"][4])


# layer 4 c3k2 stage is doubling in depth

# base repeats: 2 -> 4
cfg["backbone"][4][1] = 4

print("After: ", cfg["backbone"][4])

with open("yolo26s_c3k2_deeper.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

modified = YOLO("yolo26s_c3k2_deeper.yaml")

print("\nModified model created.")

Before: [-1, 2, 'C3k2', [512, False, 0.25]]
After:  [-1, 4, 'C3k2', [512, False, 0.25]]

Modified model created.


In [7]:
baseline_p = sum(p.numel() for p in baseline.model.parameters())
modified_p = sum(p.numel() for p in modified.model.parameters())

print(f"Baseline: {baseline_p:,}")
print(f"Modified: {modified_p:,}")
print(f"Added:    {modified_p - baseline_p:+,}")

print("\nBASELINE LAYER 4:")
print(baseline.model.model[4])

print("\nMODIFIED LAYER 4:")
print(modified.model.model[4])

Baseline: 10,009,784
Modified: 10,063,224
Added:    +53,440

BASELINE LAYER 4:
C3k2(
  (cv1): Conv(
    (conv): Conv2d(128, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
    (act): SiLU(inplace=True)
  )
  (cv2): Conv(
    (conv): Conv2d(192, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(256, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
    (act): SiLU(inplace=True)
  )
  (m): ModuleList(
    (0): Bottleneck(
      (cv1): Conv(
        (conv): Conv2d(64, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_run

In [8]:
src = baseline.model.state_dict()
dst = modified.model.state_dict()

matching = {
    k: v for k, v in src.items()
    if k in dst and v.shape == dst[k].shape
}

total_src_params = sum(v.numel() for v in src.values())
matching_params = sum(v.numel() for v in matching.values())

print(f"Matching tensors: {len(matching)} / {len(src)}")
print(f"Transferable params: {matching_params:,} / {total_src_params:,}")
print(f"Transfer ratio: {100 * matching_params / total_src_params:.2f}%")

Matching tensors: 707 / 708
Transferable params: 9,998,730 / 10,047,882
Transfer ratio: 99.51%


In [9]:
modified.load("yolo26s.pt")


#Transferred 707/720 items from pretrained weights


Transferred 707/720 items from pretrained weights


YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU()
      )
      (1): Conv(
        (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU()
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU()
        )
        (cv2): Conv(
          (conv): Conv2d(96, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(

In [11]:
results = modified(
    "https://ultralytics.com/images/bus.jpg",
    imgsz=640,
    device=0
)

results[0].show()


#lame becuase it is not pre-treained, the added bottleneck


Found https://ultralytics.com/images/bus.jpg locally at bus.jpg
image 1/1 /content/bus.jpg: 640x480 (no detections), 11.9ms
Speed: 2.1ms preprocess, 11.9ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 480)


In [ ]:
modified.train(
    data="coco128.yaml",
    epochs=15,
    imgsz=640,
    batch=16,
    device=0,
    project="cnn_yolo_lab",
    name="c3k2_depth_modified",
    seed=42
)



# only 128 pictures and used for quick pipeline experiment control
# train and valid. photos are the same here
# output is too long, log screen look below

In [13]:
import pandas as pd

path = "/content/runs/detect/cnn_yolo_lab/c3k2_depth_modified/results.csv"

df = pd.read_csv(path)
df.columns = df.columns.str.strip()

summary = df[
    [
        "epoch",
        "train/box_loss",
        "train/cls_loss",
        "metrics/precision(B)",
        "metrics/recall(B)",
        "metrics/mAP50(B)",
        "metrics/mAP50-95(B)",
    ]
]

print("=== FIRST EPOCH ===")
display(summary.head(1))

print("\n=== FINAL EPOCH ===")
display(summary.tail(1))

=== FIRST EPOCH ===


,epoch,train/box_loss,train/cls_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B)
0,1,3.195,6.45521,0.00148,0.00125,0.00006,0.00003



=== FINAL EPOCH ===


,epoch,train/box_loss,train/cls_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B)
14,15,1.9878,3.25203,0.50396,0.1966,0.21633,0.14587


## Experiment Result

Modification:
- Backbone Layer 4 C3k2 depth increased
- Internal Bottlenecks: 1 → 2
- Parameters: 10.01M → 10.06M
- Added parameters: +53,440

After 15 epochs on COCO128:
- Precision: 0.504
- Recall: 0.197
- mAP50: 0.216
- mAP50-95: 0.146

Next step:
Train the original YOLO26s with identical settings and compare results.